# Local Binary Patterns (LBP)

**LBP** is a simple yet powerful texture descriptor used in image processing and computer vision.

### Key Idea
For each pixel, compare it to its 8 neighbors. If a neighbor's value ≥ center pixel → write `1`, else → write `0`.  
Reading these 8 bits clockwise gives a number from **0 to 255** — that's the LBP code for that pixel.

---

### Outline
1. Install & Import Libraries  
2. Understand LBP Step-by-Step (Manual)  
3. Visualize LBP on a Real Image  
4. Extract the LBP Histogram (Feature Vector)  
5. Compare Textures Using LBP  
6. Use LBP for Simple Classification  

## 1. Install & Import Libraries

In [ ]:
# Run this once in Colab to make sure everything is available
!pip install scikit-image matplotlib numpy -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from skimage.feature import local_binary_pattern
from skimage import data, color, transform
from skimage.util import img_as_ubyte
from scipy.spatial.distance import euclidean

print(" All libraries imported successfully!")

## 2. LBP Step-by-Step (Manual)

Let's walk through exactly how LBP works on a tiny 3×3 patch.

```
Pixel patch:       Compare to center (128):    Binary (clockwise TL→T→TR→R→BR→B→BL→L):
┌────┬────┬────┐   ┌────┬────┬────┐            ┌───┬───┬───┐
│ 64 │150 │200 │   │ 0  │ 1  │ 1  │            │ 0 │ 1 │ 1 │
├────┼────┼────┤   ├────┼────┼────┤            ├───┼───┼───┤
│ 90 │128 │175 │   │ 0  │ C  │ 1  │  →  LBP =  │ 0 │ C │ 1 │  → 11010011 = 211
├────┼────┼────┤   ├────┼────┼────┤            ├───┼───┼───┤
│100 │210 │ 80 │   │ 0  │ 1  │ 0  │            │ 0 │ 1 │ 0 │
└────┴────┴────┘   └────┴────┴────┘            └───┴───┴───┘
```
Clockwise order: TL, T, TR, R, BR, B, BL, L  →  0,1,1,1,0,1,0,0  →  **LBP = 142**

In [ ]:
def compute_lbp_manual(patch):
    """
    Manually compute LBP for the CENTER pixel of a 3x3 patch.
    Neighbors are read clockwise: TL, T, TR, R, BR, B, BL, L
    """
    center = patch[1, 1]
    # Clockwise order of (row, col) neighbors
    neighbors = [
        patch[0, 0],  # Top-Left
        patch[0, 1],  # Top
        patch[0, 2],  # Top-Right
        patch[1, 2],  # Right
        patch[2, 2],  # Bottom-Right
        patch[2, 1],  # Bottom
        patch[2, 0],  # Bottom-Left
        patch[1, 0],  # Left
    ]
    # Compare each neighbor to center
    bits = [1 if n >= center else 0 for n in neighbors]
    # Convert bits to decimal (LSB first)
    lbp_code = sum(b * (2 ** i) for i, b in enumerate(bits))
    return center, neighbors, bits, lbp_code


# --- Demo patch ---
patch = np.array([
    [ 64, 150, 200],
    [ 90, 128, 175],
    [100, 210,  80]
], dtype=np.uint8)

center, neighbors, bits, lbp_code = compute_lbp_manual(patch)

print("3×3 Patch:")
print(patch)
print(f"\nCenter pixel value : {center}")
print(f"Neighbors (CW)     : {neighbors}")
print(f"Bits (≥ center=1)  : {bits}")
print(f"Binary string      : {''.join(map(str, bits))}")
print(f"LBP Code (decimal) : {lbp_code}")

# --- Visualize the patch ---
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle("LBP Step-by-Step on a 3×3 Patch", fontsize=14, fontweight='bold')

# Original patch
im = axes[0].imshow(patch, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("Original pixel values")
for r in range(3):
    for c in range(3):
        color_txt = 'white' if patch[r,c] < 128 else 'black'
        axes[0].text(c, r, str(patch[r,c]), ha='center', va='center',
                     fontsize=14, color=color_txt, fontweight='bold')
axes[0].set_xticks([]); axes[0].set_yticks([])

# Bit comparison
bit_patch = np.array([
    [bits[0], bits[1], bits[2]],
    [bits[7], -1,      bits[3]],
    [bits[6], bits[5], bits[4]]
])
cmap2 = plt.cm.RdYlGn
display = np.where(bit_patch == -1, 0.5, bit_patch.astype(float))
axes[1].imshow(display, cmap=cmap2, vmin=0, vmax=1)
axes[1].set_title(f"Bits (center={center})")
positions = [(0,0),(0,1),(0,2),(1,2),(2,2),(2,1),(2,0),(1,0)]
for i, (r,c) in enumerate(positions):
    axes[1].text(c, r, str(bits[i]), ha='center', va='center', fontsize=16, fontweight='bold')
axes[1].text(1, 1, 'C', ha='center', va='center', fontsize=16, fontweight='bold', color='navy')
axes[1].set_xticks([]); axes[1].set_yticks([])

# LBP result
axes[2].axis('off')
axes[2].text(0.5, 0.7, f"Binary:\n{''.join(map(str,bits))}",
             ha='center', va='center', fontsize=16, family='monospace',
             transform=axes[2].transAxes)
axes[2].text(0.5, 0.35, f"LBP Code:\n{lbp_code}",
             ha='center', va='center', fontsize=22, fontweight='bold',
             color='steelblue', transform=axes[2].transAxes)
axes[2].set_title("Result")

plt.tight_layout()
plt.show()

## 3. Visualize LBP on a Real Image

Now let's apply LBP to a real grayscale image using `scikit-image`.

**Parameters:**
- `radius` — distance from center pixel to neighbors (larger = coarser texture)
- `n_points` — number of neighbors sampled (usually `8 * radius`)
- `method='uniform'` — only patterns with ≤ 2 bit-transitions (more robust)

In [ ]:
# Load grayscale image
image = data.camera()   # classic 512×512 cameraman photo

# --- Try different radius values ---
configs = [
    (1,  8,  'radius=1 (fine texture)'),
    (2, 16,  'radius=2 (medium texture)'),
    (3, 24,  'radius=3 (coarse texture)'),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("LBP at Different Radii", fontsize=14, fontweight='bold')

axes[0,0].imshow(image, cmap='gray')
axes[0,0].set_title("Original")
axes[0,0].axis('off')
axes[1,0].axis('off')
axes[1,0].text(0.5, 0.5, 'Original Image\n512×512\nGrayscale',
               ha='center', va='center', fontsize=12, transform=axes[1,0].transAxes)

for col, (radius, n_points, title) in enumerate(configs, start=1):
    lbp = local_binary_pattern(image, n_points, radius, method='uniform')
    axes[0, col].imshow(lbp, cmap='gray')
    axes[0, col].set_title(title)
    axes[0, col].axis('off')
    # Histogram
    n_bins = int(lbp.max() + 1)
    axes[1, col].hist(lbp.ravel(), bins=n_bins, density=True, color='steelblue', alpha=0.8)
    axes[1, col].set_title(f"Histogram (P={n_points})")
    axes[1, col].set_xlabel("LBP value")
    axes[1, col].set_ylabel("Frequency")

plt.tight_layout()
plt.show()
print(" Edges and corners appear bright — they have many bit-transitions.")
print("   Flat regions appear dark — all neighbors are similar to center.")

## 4. The LBP Histogram = Your Feature Vector

After computing LBP for every pixel, you build a **histogram** of all LBP codes.  
This histogram is the final feature vector — a compact texture fingerprint of the image.

```
Image  →  LBP per pixel  →  Histogram  →  Feature vector  →  Classifier
```

In [ ]:
def get_lbp_features(image_gray, radius=1, n_points=8):
    """Compute normalized LBP histogram feature vector."""
    lbp = local_binary_pattern(image_gray, n_points, radius, method='uniform')
    n_bins = n_points + 2      # uniform LBP has n_points+2 unique patterns
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins), density=True)
    return hist, lbp


image_gray = data.camera()
features, lbp_map = get_lbp_features(image_gray, radius=1, n_points=8)

print(f"Image shape      : {image_gray.shape}")
print(f"LBP map shape    : {lbp_map.shape}")
print(f"Feature vector   : {len(features)} values")
print(f"Feature values   : {np.round(features, 4)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("From Image → LBP Map → Feature Vector", fontsize=13, fontweight='bold')

axes[0].imshow(image_gray, cmap='gray')
axes[0].set_title("① Original image")
axes[0].axis('off')

axes[1].imshow(lbp_map, cmap='nipy_spectral')
axes[1].set_title("② LBP map (each pixel = LBP code)")
axes[1].axis('off')

axes[2].bar(range(len(features)), features, color='steelblue', edgecolor='white', linewidth=0.5)
axes[2].set_title("③ Feature vector (LBP histogram)")
axes[2].set_xlabel("LBP pattern index")
axes[2].set_ylabel("Normalized frequency")

plt.tight_layout()
plt.show()

## 5. Compare Textures Using LBP

LBP is great at distinguishing textures. Let's compare different texture images and see how their LBP histograms differ.

**Similarity measure:** Euclidean distance between histograms — smaller = more similar texture.

In [ ]:
# Use built-in scikit-image textures
brick   = img_as_ubyte(color.rgb2gray(data.brick()))
grass   = img_as_ubyte(color.rgb2gray(data.grass()))
gravel  = img_as_ubyte(color.rgb2gray(data.gravel()))

textures = {'Brick': brick, 'Grass': grass, 'Gravel': gravel}
colors   = ['#E24B4A', '#639922', '#BA7517']

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
fig.suptitle("LBP Texture Comparison", fontsize=14, fontweight='bold')

hist_dict = {}
for col, (name, tex) in enumerate(textures.items()):
    hist, _ = get_lbp_features(tex, radius=2, n_points=16)
    hist_dict[name] = hist
    axes[0, col].imshow(tex, cmap='gray')
    axes[0, col].set_title(name, fontsize=13)
    axes[0, col].axis('off')
    axes[1, col].bar(range(len(hist)), hist, color=colors[col], alpha=0.85)
    axes[1, col].set_title(f"{name} — LBP histogram")
    axes[1, col].set_xlabel("LBP pattern")
    axes[1, col].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

# Pairwise distances
names = list(hist_dict.keys())
print("\n Pairwise LBP distances (Euclidean) — smaller = more similar:")
print("-" * 45)
for i in range(len(names)):
    for j in range(i+1, len(names)):
        d = euclidean(hist_dict[names[i]], hist_dict[names[j]])
        print(f"  {names[i]:8s} ↔ {names[j]:8s} : {d:.4f}")

## 6. Simple Texture Classification with LBP

Let's build a simple classifier: train on texture patches, then predict what texture an unknown patch belongs to.

We'll use **cosine similarity** between LBP histograms — a classic nearest-neighbor approach.

In [ ]:
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b) + 1e-10)

def classify_texture(query_hist, templates):
    """Nearest-neighbor classifier using cosine similarity."""
    best_name, best_score = None, -1
    scores = {}
    for name, hist in templates.items():
        score = cosine_similarity(query_hist, hist)
        scores[name] = score
        if score > best_score:
            best_score, best_name = score, name
    return best_name, scores


# Build template histograms from full texture images
templates = {}
for name, tex in textures.items():
    hist, _ = get_lbp_features(tex, radius=2, n_points=16)
    templates[name] = hist

# Test: take random crops from each texture and classify them
rng = np.random.default_rng(42)
patch_size = 64
results = []

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
fig.suptitle("LBP Nearest-Neighbor Texture Classification", fontsize=14, fontweight='bold')

for row, (true_name, tex) in enumerate(textures.items()):
    for trial in range(5):
        r = rng.integers(0, tex.shape[0] - patch_size)
        c = rng.integers(0, tex.shape[1] - patch_size)
        patch = tex[r:r+patch_size, c:c+patch_size]
        query_hist, _ = get_lbp_features(patch, radius=2, n_points=16)
        pred_name, scores = classify_texture(query_hist, templates)
        correct = pred_name == true_name
        results.append(correct)

        ax = axes[row, trial]
        ax.imshow(patch, cmap='gray')
        color_frame = '#639922' if correct else '#E24B4A'
        for spine in ax.spines.values():
            spine.set_edgecolor(color_frame)
            spine.set_linewidth(3)
        status = '✓' if correct else '✗'
        ax.set_title(f"{status} {pred_name}", fontsize=10,
                     color=color_frame, fontweight='bold')
        ax.set_xlabel(f"True: {true_name}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

accuracy = np.mean(results) * 100
print(f"\n Classification accuracy on random patches: {accuracy:.0f}%")
print("   (Green border = correct, Red = incorrect)")